# Event Dataset

## Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import plotly.express as px
import requests, time, calendar

from sklearn.feature_extraction.text import TfidfVectorizer
# cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity


## Data Exploration

In [ ]:
df = pd.read_csv("/content/Clean_Event_Dataset.csv", na_values="NaN")
df.head()

,Event_ID,Name,Categories,Environment,Location,Start_Date,End_Date,Price_Range,Family_Friendly,Description,Categories_list,Year,Month,Day,City_Key,Avg_Max_Temp,Avg_Min_Temp,Temp_max_Category,Temp_min_Category
0,1,Dubai Shopping Festival,"Festival, Entertainment",Indoor,Dubai Citywide,2026-12-15,2027-01-30,Free,True,"One of the UAE's biggest retail events, the Du...","['Festival', 'Entertainment']",2026,12,15,Dubai,25.177419,17.054839,Normal,Cold
1,2,Global Village,"Festival, Cultural, Entertainment",Outdoor,Sheikh Mohammed Bin Zayed Road Dubai,2026-10-15,2027-04-30,Low,True,Global Village is a seasonal outdoor theme par...,"['Festival', 'Cultural', 'Entertainment']",2026,10,15,Dubai,29.494140,20.227549,Normal,Normal
2,3,Abu Dhabi Art,"Exhibition, Cultural",Indoor,Manarat Al Saadiyat Abu Dhabi,2026-11-18,2026-11-22,Medium,True,Abu Dhabi Art brings together leading internat...,"['Exhibition', 'Cultural']",2026,11,18,Abu Dhabi,30.510000,23.080000,Warm,Normal
3,4,Dubai Food Festival,"Festival, Cultural",Both,Dubai Citywide,2026-02-20,2027-03-08,Free,True,The Dubai Food Festival turns the entire city ...,"['Festival', 'Cultural']",2026,2,20,Dubai,32.921160,23.282441,Warm,Normal
4,5,Abu Dhabi Grand Prix,"Sports, Entertainment",Outdoor,Yas Marina Circuit Abu Dhabi,2026-12-04,2026-12-06,High,True,The Abu Dhabi Grand Prix caps off the Formula ...,"['Sports', 'Entertainment']",2026,12,4,Abu Dhabi,26.583871,19.390323,Normal,Cold


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Event_ID         100 non-null    int64  
 1   Name             100 non-null    object 
 2   Categories       100 non-null    object 
 3   Environment      100 non-null    object 
 4   Location         100 non-null    object 
 5   Start_Date       100 non-null    object 
 6   End_Date         100 non-null    object 
 7   Price_Range      100 non-null    object 
 8   Image            0 non-null      float64
 9   Family_Friendly  100 non-null    bool   
 10  Description      100 non-null    object 
dtypes: bool(1), float64(1), int64(1), object(8)
memory usage: 8.0+ KB


In [ ]:
df.isnull().sum()

,0
Event_ID,0
Name,0
Categories,0
Environment,0
Location,0
Start_Date,0
End_Date,0
Price_Range,0
Image,100
Family_Friendly,0


In [ ]:
for col in df.columns:
    print(f"\n{col} unique values ({df[col].nunique()}):")
    print(df[col].unique())


Event_ID unique values (100):
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100]

Name unique values (100):
['Dubai Shopping Festival' 'Global Village' 'Abu Dhabi Art'
 'Dubai Food Festival' 'Abu Dhabi Grand Prix' 'Dubai World Cup'
 'Art Dubai' 'Dubai Jazz Festival' 'Abu Dhabi Food Festival'
 'Dubai Fitness Challenge' 'Sharjah Light Festival'
 'Abu Dhabi Science Festival' 'Dubai International Film Festival'
 'Taste of Dubai' 'Abu Dhabi Comedy Week' 'Dubai Design Week'
 'Sharjah Biennial' 'Dubai Marathon' 'Abu Dhabi Marathon'
 'UAE National Day Celebrations' "New Year's Eve Fireworks Dubai"
 "New Year's Eve Abu Dhabi" 'Dubai Tennis Cham

In [ ]:
print("\nDuplicate Event_IDs:", df['Event_ID'].duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())


Duplicate Event_IDs: 0
Duplicate rows: 0


In [ ]:
#distribution later on

##Preprocessing

In [ ]:
df_clean = df.copy()

df_clean['Categories_list'] = df_clean['Categories'].apply(
    lambda x: [c.strip() for c in x.split(',')] if ',' in x else [x]
)

df_clean['Start_Date'] = pd.to_datetime(df_clean['Start_Date'], errors='coerce')
df_clean['End_Date'] = pd.to_datetime(df_clean['End_Date'], errors='coerce')

df_clean = df_clean.drop(columns=['Image'])

In [ ]:
df_clean['Year'] = df_clean['Start_Date'].dt.year
df_clean['Month'] = df_clean['Start_Date'].dt.month
df_clean['Day'] = df_clean['Start_Date'].dt.day


In [ ]:
df_clean[['Year','Month','Day']].head()

,Year,Month,Day
0,2026,12,15
1,2026,10,15
2,2026,11,18
3,2026,2,20
4,2026,12,4


**Selected columns:**

Event_ID, Name, Description, Categories, Environment, Location, Start_Date, End_Date, Price_Range (plus derived: Year, Month, Day, Price_Min, Price_Max)

**Purpose:**


This dataset provides the core event catalog for the tourist recommendation app — it supplies what events exist, where, when, what category/environment they fall under, and their price range, which feeds into the recommendation and filtering logic (e.g. matching events to a visitor's city, budget, and date range).


**Limitation and further improvement**

This Dataset needs a more specifc number for its price range and a tag for each place if it is kids friendly and expected weather. In addition to a rating and popularity column. Will collect further features for this dataset.

## Feature Engineering

In [ ]:
city_coords = {
    'Dubai': (25.2048, 55.2708),
    'Abu Dhabi': (24.4539, 54.3773),
    'Sharjah': (25.3463, 55.4209),
    'Ras Al Khaimah': (25.7895, 55.9432),
    'Fujairah': (25.1288, 56.3265),
}

def get_city(location):
    for city in city_coords:
        if city.lower() in location.lower():
            return city
    return 'Dubai'

df_clean['City_Key'] = df_clean['Location'].apply(get_city)

# Collect every (city, year, month) the event range touches
unique_months = set()
for _, row in df_clean.iterrows():
    for period in pd.period_range(row['Start_Date'], row['End_Date'], freq='M'):
        unique_months.add((row['City_Key'], period.year, period.month))

print(f"Unique city-month lookups needed: {len(unique_months)}")

Unique city-month lookups needed: 42


In [ ]:
weather_cache = {}

def fetch_month_climate(lat, lon, year, month):
    ref_year = min(year, pd.Timestamp.now().year - 1)
    last_day = calendar.monthrange(ref_year, month)[1]
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": f"{ref_year}-{month:02d}-01",
        "end_date": f"{ref_year}-{month:02d}-{last_day}",
        "daily": "temperature_2m_max,temperature_2m_min",
        "timezone": "auto",
    }
    r = requests.get(url, params=params)
    data = r.json().get("daily", {})
    return {
        "avg_max_temp": sum(data.get("temperature_2m_max", [0])) / max(len(data.get("temperature_2m_max", [1])), 1),
        "avg_min_temp": sum(data.get("temperature_2m_min", [0])) / max(len(data.get("temperature_2m_min", [1])), 1),
    }

for city, year, month in unique_months:
    key = (city, year, month)
    if key not in weather_cache:
        lat, lon = city_coords[city]
        weather_cache[key] = fetch_month_climate(lat, lon, year, month)
        time.sleep(0.2)

print(f"API calls made: {len(weather_cache)}")

API calls made: 42


In [ ]:
def get_event_weather(row):
    months = pd.period_range(row['Start_Date'], row['End_Date'], freq='M')
    records = [weather_cache[(row['City_Key'], p.year, p.month)] for p in months]
    return pd.Series({
        "avg_max_temp": sum(r['avg_max_temp'] for r in records) / len(records),
        "avg_min_temp": sum(r['avg_min_temp'] for r in records) / len(records),
    })

df_clean[['Avg_Max_Temp', 'Avg_Min_Temp']] = df_clean.apply(get_event_weather, axis=1)
df_clean[['Avg_Max_Temp', 'Avg_Min_Temp']]

,Avg_Max_Temp,Avg_Min_Temp
0,25.177419,17.054839
1,29.494140,20.227549
2,30.510000,23.080000
3,32.921160,23.282441
4,26.583871,19.390323
...,...,...
95,34.912903,25.922581
96,39.053333,28.086667
97,30.973333,20.276667
98,35.430000,23.230000


In [ ]:
def temp_category(avg_max):
    if avg_max >= 38:
        return "Hot"
    elif avg_max >= 30:
        return "Warm"
    elif avg_max >= 20:
        return "Normal"
    else:
        return "Cold"

df_clean['Temp_max_Category'] = df_clean['Avg_Max_Temp'].apply(temp_category)
df_clean['Temp_min_Category'] = df_clean['Avg_Min_Temp'].apply(temp_category)

In [ ]:
df_clean.isnull().sum()

,0
Event_ID,0
Name,0
Categories,0
Environment,0
Location,0
Start_Date,0
End_Date,0
Price_Range,0
Family_Friendly,0
Description,0


In [ ]:
df_clean[['Name', 'Avg_Max_Temp', 'Avg_Min_Temp', 'Temp_max_Category','Temp_min_Category']]

,Name,Avg_Max_Temp,Avg_Min_Temp,Temp_max_Category,Temp_min_Category
0,Dubai Shopping Festival,25.177419,17.054839,Normal,Cold
1,Global Village,29.494140,20.227549,Normal,Normal
2,Abu Dhabi Art,30.510000,23.080000,Warm,Normal
3,Dubai Food Festival,32.921160,23.282441,Warm,Normal
4,Abu Dhabi Grand Prix,26.583871,19.390323,Normal,Cold
...,...,...,...,...,...
95,Beautyworld Middle East,34.912903,25.922581,Warm,Normal
96,Automechanika Dubai,39.053333,28.086667,Hot,Normal
97,The Big 5 Dubai,30.973333,20.276667,Warm,Normal
98,Arabian Travel Market,35.430000,23.230000,Warm,Normal


In [ ]:
df_clean.to_csv("Clean_Event_Dataset.csv", index=False)

## Distination Content

In [ ]:
df_clean.columns

Index(['Event_ID', 'Name', 'Categories', 'Environment', 'Location',
       'Start_Date', 'End_Date', 'Price_Range', 'Family_Friendly',
       'Description', 'Categories_list', 'Year', 'Month', 'Day', 'City_Key',
       'Avg_Max_Temp', 'Avg_Min_Temp', 'Temp_max_Category',
       'Temp_min_Category'],
      dtype='object')

In [ ]:
catalog_columns = ['Name','City_Key', 'Description','Categories_list','Categories','Price_Range', 'Environment', 'Location','Start_Date', 'End_Date','Temp_max_Category', 'Temp_min_Category','Family_Friendly']
catalog_df = df_clean[catalog_columns].copy()

In [ ]:
catalog_df.head(1)

,Name,City_Key,Description,Categories_list,Categories,Price_Range,Environment,Location,Start_Date,End_Date,Temp_max_Category,Temp_min_Category,Family_Friendly
0,Dubai Shopping Festival,Dubai,"One of the UAE's biggest retail events, the Du...","[Festival, Entertainment]","Festival, Entertainment",Free,Indoor,Dubai Citywide,2026-12-15,2027-01-30,Normal,Cold,True


In [ ]:
catalog_df['destination_content'] = (
    catalog_df['Name'].fillna(' ')+' '+
    catalog_df['Description'].fillna(' ')+' '+
    catalog_df['Categories'].fillna(' ')+' '+
    catalog_df['Environment'].fillna(' ')+' '+
    catalog_df['Location'].fillna(' ')
    ).str.lower()

In [ ]:
catalog_df['destination_content']

,destination_content
0,dubai shopping festival one of the uae's bigge...
1,global village global village is a seasonal ou...
2,abu dhabi art abu dhabi art brings together le...
3,dubai food festival the dubai food festival tu...
4,abu dhabi grand prix the abu dhabi grand prix ...
...,...
95,beautyworld middle east beautyworld middle eas...
96,automechanika dubai automechanika dubai is an ...
97,the big 5 dubai the big 5 dubai is a major con...
98,arabian travel market the arabian travel marke...


In [ ]:
catalog_df.columns

Index(['Name', 'City_Key', 'Description', 'Categories_list', 'Categories',
       'Price_Range', 'Environment', 'Location', 'Start_Date', 'End_Date',
       'Temp_max_Category', 'Temp_min_Category', 'Family_Friendly',
       'destination_content'],
      dtype='object')

## Fall Back & Filtering

In [ ]:
def has_overlap(cats, preferences):
    if not isinstance(cats, list):
        return False
    return any(c in preferences for c in cats)

def filtering_fallback(catalog_df, n_recommendation, Start_Date, End_Date, environment=None, city=None, budget=None, family_friendly=None, temp_pref=None, event_preference=None):

    base = catalog_df[
        (catalog_df['Start_Date'] <= End_Date) &
        (catalog_df['End_Date'] >= Start_Date)
    ]

    if family_friendly is not None:
        base = base[base['Family_Friendly'] == family_friendly]

    stage_names = ['strict_match', 'relaxed_weather', 'relaxed_budget', 'expanded_city', 'relaxed_event']
    candidates_df = pd.DataFrame()

    for stage_name in stage_names:
        if len(candidates_df) >= n_recommendation:
            break

        remaining = base[~base.index.isin(candidates_df.index)]
        mask = pd.Series(True, index=remaining.index)

        if environment is not None:
            mask &= remaining['Environment'].apply(
                lambda env: env == 'Both' or env in environment
            )

        if city is not None and stage_name not in ['expanded_city', 'relaxed_event']:
            mask &= remaining['City_Key'].isin(city)

        if budget is not None and stage_name in ['strict_match', 'relaxed_weather']:
            mask &= remaining['Price_Range'].isin(budget)

        # Weather: one consistent field (temp_pref) checked against both max/min category columns
        if temp_pref is not None and stage_name == 'strict_match':
            mask &= (
                remaining['Temp_max_Category'].isin(temp_pref) |
                remaining['Temp_min_Category'].isin(temp_pref)
            )

        if event_preference is not None and stage_name != 'relaxed_event':
            mask &= remaining['Categories'].apply(lambda cats: any(e in cats for e in event_preference))

        matches = remaining[mask].copy()
        matches['fallback_stage'] = stage_name
        candidates_df = pd.concat([candidates_df, matches])

    return candidates_df.head(n_recommendation)

##Vectorizing the items

In [ ]:
def build_event_vectorizer(events_df):
    event_text = catalog_df['destination_content']
    tfidf = TfidfVectorizer(stop_words='english')
    event_vectors = tfidf.fit_transform(event_text)
    return tfidf, event_vectors

tfidf_vectorizer, all_event_vectors = build_event_vectorizer(catalog_df)


## Similarity Function

In [ ]:
def build_visitor_text(visitor_profile):
    activities = ', '.join(visitor_profile.get('Activity_Preferences', []))
    activity_other = visitor_profile.get('Activity_Other', '') or ''
    cuisines = ', '.join(visitor_profile.get('Cuisine_Preferences', []))
    cuisine_other = visitor_profile.get('Cuisine_Other', '') or ''
    crowd_pref = visitor_profile.get('Crowdedness_Preference', '')
    family = visitor_profile.get('Family_Friendly', False)
    environments = ', '.join(visitor_profile.get('Environment', []))
    special_note = visitor_profile.get('special_note', '') or ''

    visitor_text = (
        f"I am interested in {activities}"
        f"{' and ' + activity_other if activity_other else ''} activities. "
        f"I enjoy {cuisines}"
        f"{' and ' + cuisine_other if cuisine_other else ''} food. "
        f"I prefer {environments} settings with a {crowd_pref.lower() if crowd_pref else ''} crowd level. "
        f"{'I am traveling with kids and prefer family-friendly options.' if family else 'I am not traveling with kids.'} "
        f"{special_note}"
    )
    return visitor_text

In [ ]:
def compute_similarity(candidates_df, visitor_profile, tfidf_vectorizer, all_event_vectors):
    visitor_text = build_visitor_text(visitor_profile)
    visitor_vector = tfidf_vectorizer.transform([visitor_text])

    candidate_indices = candidates_df.index
    candidate_vectors = all_event_vectors[candidate_indices]

    similarity_scores = cosine_similarity(candidate_vectors, visitor_vector).flatten()

    candidates_df = candidates_df.copy()
    candidates_df['similarity_score'] = similarity_scores
    return candidates_df


## Reason behind Recommendation

In [ ]:
def build_recommendation_reason(row, visitor_profile):
    reasons = {
        'strict_match': 'Matches your city, budget, and weather preference',
        'relaxed_weather': 'Matches your city and budget (weather preference relaxed)',
        'relaxed_budget': 'Matches your city (budget relaxed)',
        'expanded_city': 'Recommended from other locations to meet your request',
        'relaxed_event': 'Recommended based on your general preferences',
    }
    reason = reasons.get(row['fallback_stage'], 'Recommended based on your profile')

    extras = []
    if row.get('Family_Friendly') is True:
        extras.append('family-friendly')

    prefs = set(p.lower() for p in visitor_profile.get('Activity_Preferences', []))
    categories = set(c.lower() for c in row['Categories'])
    matched_activities = prefs & categories
    if matched_activities:
        extras.append(f"matches your interest in {', '.join(matched_activities)}")

    if row.get('date_in_range', True):
        extras.append('falls within your trip dates')

    if extras:
        reason += " (" + ", ".join(extras) + ")"

    return reason

## Compiled Function

In [ ]:
def recommend_events(catalog_df, visitor_profile, tfidf_vectorizer, all_event_vectors):
    n_recommendation = visitor_profile['Num_Recommendations']

    # 1. Filter candidates using fallback stages
    candidates_df = filtering_fallback(
      catalog_df,
      n_recommendation=n_recommendation,
      Start_Date=visitor_profile['Trip_Start_Date'],
      End_Date=visitor_profile['Trip_End_Date'],
      environment=visitor_profile.get('Environment'),
      city=visitor_profile.get('City'),
      budget=visitor_profile.get('budget'),
      family_friendly=visitor_profile.get('Family_Friendly'),
      temp_pref=visitor_profile.get('Temp_Pref'),
      event_preference=visitor_profile.get('Event_Preferences')
    )
    # 2. Compute similarity score
    candidates_df = compute_similarity(candidates_df, visitor_profile, tfidf_vectorizer, all_event_vectors)

    # 3. Apply fallback-stage penalty
    stage_penalty = {
        'strict_match': 1.0,
        'relaxed_weather': 0.80,
        'relaxed_budget': 0.75,
        'expanded_city': 0.6,
        'relaxed_event': 0.5
    }
    candidates_df['final_score'] = candidates_df['similarity_score'] * candidates_df['fallback_stage'].map(stage_penalty)

    # 4. Select Top-N
    final_recommendations = candidates_df.sort_values('final_score', ascending=False).head(n_recommendation).copy()

    # 5. Add visitor_id and recommendation reason
    final_recommendations['visitor_id'] = visitor_profile['User_ID']
    final_recommendations['recommendation_reason'] = final_recommendations.apply(
        lambda row: build_recommendation_reason(row, visitor_profile), axis=1
    )

    # 6. Round scoring columns
    final_recommendations[['similarity_score', 'final_score']] = final_recommendations[['similarity_score', 'final_score']].round(3)

    # 7. Select and order required columns
    required_columns = [
        'visitor_id', 'Name', 'Categories', 'Location',
        'Start_Date', 'End_Date', 'Price_Range', 'similarity_score',
        'fallback_stage', 'final_score', 'recommendation_reason', 'Description'
    ]
    return final_recommendations[required_columns].sort_values(by='final_score', ascending=False).reset_index(drop=True)

## Demo User Profile based on the google form

In [ ]:
visitor_profile_A= {
    "User_ID": "v001",
    "Trip_Start_Date": pd.Timestamp("2026-10-1"),
    "Trip_End_Date": pd.Timestamp("2026-11-15"),
    "City": ["Abu Dhabi"],
    "budget": ["Free","Low","Medium","High"],
    "Event_Preferences": ["Cultural","Exhibition"],
    "Cuisine_Preferences": "",
    "Cuisine_Other": "",
    "Environment": ["Indoor"],
    "Family_Friendly": True,
    "Temp_Pref": None,
    "Num_Recommendations": 5,
    "special_note":"I want to learn about the culture and do some shopping"
}

In [ ]:
recommendation_A=recommend_events(catalog_df, visitor_profile_A, tfidf_vectorizer, all_event_vectors)

In [ ]:
recommendation_A

,visitor_id,Name,Categories,Location,Start_Date,End_Date,Price_Range,similarity_score,fallback_stage,final_score,recommendation_reason,Description
0,v001,Dubai Food Festival,"Festival, Cultural",Dubai Citywide,2026-02-20,2027-03-08,Free,0.270,expanded_city,0.162,Recommended from other locations to meet your ...,The Dubai Food Festival turns the entire city ...
1,v001,Abu Dhabi Food Festival,"Festival, Cultural",Abu Dhabi Citywide,2026-11-05,2026-11-21,Free,0.107,strict_match,0.107,"Matches your city, budget, and weather prefere...",The Abu Dhabi Food Festival spreads culinary e...
2,v001,Abu Dhabi Science Festival,"Exhibition, Cultural",Various Abu Dhabi Locations,2026-11-12,2026-11-21,Free,0.010,strict_match,0.010,"Matches your city, budget, and weather prefere...",The Abu Dhabi Science Festival fills venues ac...
3,v001,Abu Dhabi Fitness Expo,"Exhibition, Sports",ADNEC Abu Dhabi,2026-11-06,2026-11-08,Low,0.010,strict_match,0.010,"Matches your city, budget, and weather prefere...",The Abu Dhabi Fitness Expo brings together hea...
4,v001,Louvre Abu Dhabi Exhibitions,"Exhibition, Cultural",Louvre Abu Dhabi,2026-01-01,2026-12-31,Medium,0.009,strict_match,0.009,"Matches your city, budget, and weather prefere...",Louvre Abu Dhabi's rotating exhibitions bring ...


In [ ]:
visitor_profile_B= {
    "User_ID": "v001",
    "Trip_Start_Date": pd.Timestamp("2026-12-1"),
    "Trip_End_Date": pd.Timestamp("2026-12-15"),
    "City": ["Abu Dhabi"],
    "budget": ["Free","Low","Medium"],
    "Event_Preferences": ["Concerts"],
    "Cuisine_Preferences": "",
    "Cuisine_Other": "",
    "Environment": ["Outdoor","Indoor"],
    "Family_Friendly": True,
    "Temp_Pref": ["Warm"],
    "Num_Recommendations": 5,
    "special_note":"I would love to party and dance and hear some idols sing"
}

In [ ]:
recommendation_B=recommend_events(catalog_df,visitor_profile_B, tfidf_vectorizer, all_event_vectors)
recommendation_B

,visitor_id,Name,Categories,Location,Start_Date,End_Date,Price_Range,similarity_score,fallback_stage,final_score,recommendation_reason,Description
0,v001,Dubai Food Festival,"Festival, Cultural",Dubai Citywide,2026-02-20,2027-03-08,Free,0.271,relaxed_event,0.136,Recommended based on your general preferences ...,The Dubai Food Festival turns the entire city ...
1,v001,Global Village,"Festival, Cultural, Entertainment",Sheikh Mohammed Bin Zayed Road Dubai,2026-10-15,2027-04-30,Low,0.100,relaxed_event,0.050,Recommended based on your general preferences ...,Global Village is a seasonal outdoor theme par...
2,v001,Dubai Shopping Festival,"Festival, Entertainment",Dubai Citywide,2026-12-15,2027-01-30,Free,0.011,relaxed_event,0.005,Recommended based on your general preferences ...,"One of the UAE's biggest retail events, the Du..."
3,v001,Dubai International Film Festival,"Festival, Cultural",Madinat Jumeirah Dubai,2026-12-09,2026-12-16,Medium,0.010,relaxed_event,0.005,Recommended based on your general preferences ...,The Dubai International Film Festival celebrat...
4,v001,Abu Dhabi Grand Prix,"Sports, Entertainment",Yas Marina Circuit Abu Dhabi,2026-12-04,2026-12-06,High,0.010,relaxed_event,0.005,Recommended based on your general preferences ...,The Abu Dhabi Grand Prix caps off the Formula ...


In [ ]:
visitor_profile_C = {
    "User_ID": "v001",
    "Trip_Start_Date": pd.Timestamp("2026-10-01"),
    "Trip_End_Date": pd.Timestamp("2026-11-15"),
    "City": ["Abu Dhabi"],
    "budget": ["Free", "Low", "Medium", "High"],
    "Event_Preferences": ["Cultural", "Exhibition"],
    "Activity_Preferences": ["Cultural", "Exhibition"],
    "Cuisine_Preferences": [],
    "Cuisine_Other": "",
    "Environment": ["Indoor"],
    "Family_Friendly": True,
    "Temp_Pref": ["Cold"],
    "Num_Recommendations": 6,
    "special_note": "I want to learn about the culture and do some shopping"
}

In [ ]:
recommendation_C=recommend_events(catalog_df,visitor_profile_C, tfidf_vectorizer, all_event_vectors)
recommendation_C

,visitor_id,Name,Categories,Location,Start_Date,End_Date,Price_Range,similarity_score,fallback_stage,final_score,recommendation_reason,Description
0,v001,Dubai Food Festival,"Festival, Cultural",Dubai Citywide,2026-02-20,2027-03-08,Free,0.277,expanded_city,0.166,Recommended from other locations to meet your ...,The Dubai Food Festival turns the entire city ...
1,v001,Abu Dhabi Food Festival,"Festival, Cultural",Abu Dhabi Citywide,2026-11-05,2026-11-21,Free,0.121,relaxed_weather,0.097,Matches your city and budget (weather preferen...,The Abu Dhabi Food Festival spreads culinary e...
2,v001,Abu Dhabi Science Festival,"Exhibition, Cultural",Various Abu Dhabi Locations,2026-11-12,2026-11-21,Free,0.040,relaxed_weather,0.032,Matches your city and budget (weather preferen...,The Abu Dhabi Science Festival fills venues ac...
3,v001,Louvre Abu Dhabi Exhibitions,"Exhibition, Cultural",Louvre Abu Dhabi,2026-01-01,2026-12-31,Medium,0.038,relaxed_weather,0.030,Matches your city and budget (weather preferen...,Louvre Abu Dhabi's rotating exhibitions bring ...
4,v001,Abu Dhabi Fitness Expo,"Exhibition, Sports",ADNEC Abu Dhabi,2026-11-06,2026-11-08,Low,0.025,relaxed_weather,0.020,Matches your city and budget (weather preferen...,The Abu Dhabi Fitness Expo brings together hea...
5,v001,Dubai Design Week,"Exhibition, Cultural",Dubai Design District,2026-11-08,2026-11-13,Free,0.025,expanded_city,0.015,Recommended from other locations to meet your ...,Dubai Design Week is the region's largest crea...


In [ ]:
import joblib

# Save
joblib.dump(tfidf_vectorizer, 'event_tfidf_vectorizer.pkl')
joblib.dump(all_event_vectors, 'all_event_vectors.pkl')

['all_event_vectors.pkl']